# WAFT (waft-a2, dav2) - demo fine-tuning in Colab

Pasi: clone repo -> instalare dependinte -> incarcare checkpoint + encoder dav2 -> incarcare set propriu de date (poze + ground truth) -> fine-tuning cateva pasi -> vizualizare flux optic inainte/dupa.

## 1. Clonare repo
Inlocuieste `REPO_URL` cu URL-ul repo-ului tau (cel pe care l-ai facut push cu acest folder minimal).

In [ ]:
REPO_URL = "https://github.com/<user>/<repo>.git"  # <-- pune aici URL-ul tau

!git clone $REPO_URL waft-demo
%cd waft-demo

## 2. Instalare dependinte

In [ ]:
!pip install -q -r requirements.txt

## 3. Checkpoint WAFT de pornire
Incarca fisierul `.pth` (checkpoint-ul waft-a2/dav2, ex. cel din Google Drive-ul proiectului original) direct din calculator.

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)

from google.colab import files
uploaded = files.upload()  # selecteaza checkpoint-ul .pth
for fname in uploaded:
    os.rename(fname, os.path.join('checkpoints', 'sintel-gm-final.pth'))
print(os.listdir('checkpoints'))

## 4. Greutatile encoder-ului dav2 (Depth Anything V2, vits)
Incearca intai descarcarea automata de pe Hugging Face (link public al proiectului Depth-Anything-V2). Daca nu merge (retea/permisiuni), incarca manual fisierul cu celula de mai jos.

In [ ]:
import os
os.makedirs('depth-anything-ckpts', exist_ok=True)
url = "https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth"
dst = 'depth-anything-ckpts/depth_anything_v2_vits.pth'
!wget -q --show-progress -O $dst $url
print(os.path.getsize(dst) if os.path.exists(dst) else 'download failed - incarca manual mai jos')

In [ ]:
# Ruleaza doar daca descarcarea automata de mai sus a esuat
from google.colab import files
uploaded = files.upload()  # selecteaza depth_anything_v2_vits.pth
for fname in uploaded:
    os.rename(fname, 'depth-anything-ckpts/depth_anything_v2_vits.pth')

## 5. Setul tau de date (poze + ground truth)
Incarca o arhiva `.zip` cu structura:
```
image1/000000.png ...
image2/000000.png ...
flow/000000.flo ...
```
(vezi README.md pentru detalii despre formatul `.flo`).

In [ ]:
import os, zipfile
from google.colab import files

os.makedirs('data/custom', exist_ok=True)
uploaded = files.upload()  # selecteaza arhiva .zip cu image1/ image2/ flow/
for fname in uploaded:
    with zipfile.ZipFile(fname) as z:
        z.extractall('data/custom')

print(os.listdir('data/custom'))

## 6. Fine-tuning
Ruleaza cativa pasi de antrenare pornind de la checkpoint, pe setul tau de date.

In [ ]:
!python finetune_demo.py \
    --cfg config/a2/dav2/sintel-gm.json \
    --ckpt checkpoints/sintel-gm-final.pth \
    --data_dir data/custom \
    --steps 100 \
    --out_ckpt checkpoints/finetuned.pth

## 7. Vizualizare inainte / dupa fine-tuning

In [ ]:
import matplotlib.pyplot as plt
import cv2

before = cv2.cvtColor(cv2.imread('demo_out/flow_before.jpg'), cv2.COLOR_BGR2RGB)
after = cv2.cvtColor(cv2.imread('demo_out/flow_after.jpg'), cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(before); ax[0].set_title('Flux optic - inainte de fine-tuning'); ax[0].axis('off')
ax[1].imshow(after); ax[1].set_title('Flux optic - dupa fine-tuning'); ax[1].axis('off')
plt.show()